# Assistente Médico com LangChain + LangGraph — Tech Challenge 3

## 1) Configuração e carregamento do LLM treinado

Tentamos carregar o adaptador LoRA salvo em `/content/lora_model` sobre o modelo base `unsloth/llama-3-8b-bnb-4bit` (mesmos nomes usados no notebook de fine-tuning). Se isso falhar por qualquer motivo — sem GPU disponível, arquivo do adaptador ausente, etc. — `llm` fica `None` e os nós de geração mais à frente caem num modo offline com respostas simuladas, para que o notebook continue executável e didático mesmo fora do Colab com GPU.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters langgraph faiss-cpu sentence-transformers python-dotenv pandas
!pip install -q transformers peft accelerate bitsandbytes
!pip install grandalf
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 11.9 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import logging
from datetime import datetime
from typing import List, Dict, Optional
from typing_extensions import TypedDict
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from dotenv import load_dotenv
from langgraph.graph import StateGraph, END
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

load_dotenv()

BASE_MODEL = "unsloth/llama-3-8b-bnb-4bit"
LORA_PATH = "/content/drive/MyDrive/Tech_Challenge_3/lora_model"

llm = None
USE_LLM_FINETUNED = False

# Suprime os avisos de verbosidade do transformers (max_new_tokens/max_length,
# clean_up_tokenization_spaces etc.)
import warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

try:
    import torch
    from peft import PeftModel
    from langchain_huggingface import HuggingFacePipeline

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(LORA_PATH)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )

    modelo_finetuned = PeftModel.from_pretrained(base_model, LORA_PATH)
    modelo_finetuned.generation_config.max_length = None

    gerador = pipeline(
        "text-generation",
        model=modelo_finetuned,
        tokenizer=tokenizer,
        max_new_tokens=150,
        max_length=None,
        do_sample=True,
        temperature=0.4,
        top_p=0.9,
        repetition_penalty=1.3,
        no_repeat_ngram_size=3,
        return_full_text=False,
        clean_up_tokenization_spaces=False,
    )
    llm = HuggingFacePipeline(pipeline=gerador)
    USE_LLM_FINETUNED = True
    print("LLM fine-tuned carregado de", LORA_PATH)
    print("Device do modelo:", next(base_model.parameters()).device)
except Exception as e:
    print(f"Não foi possivel carregar o modelo fine-tuned ({e}). Rodando em modo offline/mock.")


config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.70GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

LLM fine-tuned carregado de /content/drive/MyDrive/Tech_Challenge_3/lora_model
Device do modelo: cuda:0


## 2) Embeddings e base de protocolos internos (RAG)

In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

PROTOCOLOS_DIR = "/content/drive/MyDrive/Tech_Challenge_3/protocolos"

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=80)
docs: List[Document] = []

pdfs_encontrados = (
    os.path.isdir(PROTOCOLOS_DIR)
    and any(f.lower().endswith(".pdf") for f in os.listdir(PROTOCOLOS_DIR))
)

if pdfs_encontrados:
    for nome_arquivo in sorted(os.listdir(PROTOCOLOS_DIR)):
        if not nome_arquivo.lower().endswith(".pdf"):
            continue
        caminho = os.path.join(PROTOCOLOS_DIR, nome_arquivo)
        paginas = PyPDFLoader(caminho).load()
        for pagina in paginas:
            pagina.metadata["source"] = nome_arquivo
        docs.extend(splitter.split_documents(paginas))
    print(f"{len(docs)} trechos carregados de PDFs em {PROTOCOLOS_DIR}")
else:
    print(f"Nenhum PDF encontrado em {PROTOCOLOS_DIR} -- usando protocolos de exemplo (fallback).")
    protocolos_texto = {
        "protocolo_fraturas_v3.pdf": (
            "Protocolo de fratura de femur: priorizar reducao cirurgica em ate 48h em idosos, "
            "com avaliacao de risco cardiovascular pre-operatorio."
        ),
        "protocolo_endocrino_v2.pdf": (
            "Protocolo de diabetes tipo 2: revisar hemograma e funcao renal antes de ajustar dose "
            "de metformina em pacientes acima de 65 anos."
        ),
        "politica_ia_hospital.pdf": (
            "Politica geral do hospital: qualquer sugestao de tratamento gerada por IA deve ser "
            "validada por um medico responsavel antes de ser aplicada ao paciente."
        ),
    }
    for fonte, texto in protocolos_texto.items():
        for chunk in splitter.split_text(texto):
            docs.append(Document(page_content=chunk, metadata={"source": fonte}))

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})


1159 trechos carregados de PDFs em /content/drive/MyDrive/Tech_Challenge_3/protocolos


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 3) Base estruturada de prontuários (simulada)

Representa o "registro de pacientes" citado no desafio — em produção seria a conexão com o sistema real de prontuário eletrônico do hospital. Aqui usamos SQLite em memória para manter o notebook independente de integrações.

In [5]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
prontuarios = pd.DataFrame([
    {"patient_id": "P001", "iniciais": "J.S.", "idade": 67, "historico": "Hipertensao, diabetes tipo 2",
     "exames_pendentes": "Hemograma completo", "medicamentos_atuais": "Metformina, Losartana"},
    {"patient_id": "P002", "iniciais": "M.A.", "idade": 45, "historico": "Sem comorbidades relevantes",
     "exames_pendentes": "", "medicamentos_atuais": "Nenhum"},
])
prontuarios.to_sql("prontuarios", conn, index=False, if_exists="replace")

def consultar_prontuario(patient_id: str) -> Optional[dict]:
    cur = conn.execute("SELECT * FROM prontuarios WHERE patient_id = ?", (patient_id,))
    colunas = [c[0] for c in cur.description]
    linha = cur.fetchone()
    return dict(zip(colunas, linha)) if linha else None


## 4) Estado compartilhado

In [6]:
class EstadoAssistente(TypedDict, total=False):
    patient_id: str
    pergunta_medico: str
    dados_paciente: Optional[dict]
    alerta: Optional[str]
    intencao: str
    contexto_protocolo: Optional[str]
    fontes_protocolo: Optional[List[str]]
    sugestao: Optional[str]
    flag_revisao_obrigatoria: Optional[bool]


## 5) Log de auditoria

In [7]:
from datetime import datetime, timezone

logging.basicConfig(
    filename="/content/drive/MyDrive/Tech_Challenge_3/assistente_medico_audit.log",
    level=logging.INFO,
    format="%(message)s",
)
audit_logger = logging.getLogger("audit")

def log_auditoria(evento: dict):
    evento = dict(evento)
    evento["timestamp"] = datetime.now(timezone.utc).isoformat()
    audit_logger.info(json.dumps(evento, ensure_ascii=False))


## 6) Nós (funções)

Cada nó mapeia direto para um item:

- `carregar_paciente` / `verificar_exames_pendentes` — consultam o prontuário e emitem alerta se houver exame pendente.
- `classificar_intencao` — roteamento simples: pergunta clínica segue para o fluxo com RAG + LLM; qualquer outra coisa (ex.: pergunta administrativa) é tratada fora do escopo do assistente.
- `buscar_protocolo` — nó de recuperação (retrieve) do RAG.
- `gerar_sugestao` — nó de geração, usando o LLM fine-tuned (ou fallback offline).
- `validar_seguranca` — filtro de saída que nunca deixa passar uma prescrição direta sem marcar para revisão humana.
- `resposta_fora_do_escopo` — nó final da rota alternativa.

In [8]:
def carregar_paciente(estado: EstadoAssistente) -> EstadoAssistente:
    dados = consultar_prontuario(estado["patient_id"])
    log_auditoria({"etapa": "carregar_paciente", "patient_id": estado["patient_id"], "encontrado": dados is not None})
    return {"dados_paciente": dados}


def verificar_exames_pendentes(estado: EstadoAssistente) -> EstadoAssistente:
    dados = estado.get("dados_paciente") or {}
    pendentes = dados.get("exames_pendentes", "")
    alerta = f"Exame pendente: {pendentes}" if pendentes else None
    log_auditoria({"etapa": "verificar_exames", "patient_id": estado["patient_id"], "alerta": alerta})
    return {"alerta": alerta}


In [9]:
def classificar_intencao(estado: EstadoAssistente) -> EstadoAssistente:
    pergunta = estado.get("pergunta_medico", "").lower()
    termos_clinicos = ["diagnostico", "diagnóstico", "sintoma", "tratamento", "conduta", "protocolo"]
    intencao = "duvida_clinica" if any(t in pergunta for t in termos_clinicos) else "fora_de_escopo"
    log_auditoria({"etapa": "classificar_intencao", "patient_id": estado["patient_id"], "intencao": intencao})
    return {"intencao": intencao}


def rota_intencao(estado: EstadoAssistente) -> str:
    return estado.get("intencao", "fora_de_escopo")


O nó abaixo é o **retrieve** do RAG: busca no índice de protocolos os trechos mais relevantes para a pergunta do médico e guarda tanto o texto quanto a fonte de cada trecho — a fonte é o que sustenta a explainability da resposta final.

In [10]:
def buscar_protocolo(estado: EstadoAssistente) -> EstadoAssistente:
    documentos = retriever.invoke(estado["pergunta_medico"])
    contexto = "\n".join(d.page_content for d in documentos)
    fontes = list({d.metadata["source"] for d in documentos})
    log_auditoria({"etapa": "buscar_protocolo", "patient_id": estado["patient_id"], "fontes": fontes})
    return {"contexto_protocolo": contexto, "fontes_protocolo": fontes}


O nó de geração usa o padrão `prompt | llm`. A diferença é o próprio `llm`: aqui é o pipeline do modelo fine-tuned (carregado na primeira célula), não o `ChatOpenAI`. Se o modelo não estiver disponível, caímos num texto de fallback baseado só no contexto recuperado.

In [11]:
ALPACA_TEMPLATE = (
    "Below is an instruction that describes a task, paired with an input that provides "
    "further context. Write a response that appropriately completes the request.\n\n"
    "### Instruction:\n{instruction}\n\n"
    "### Input:\n{input}\n\n"
    "### Response:\n"
)

INSTRUCAO_DIAGNOSTICO = (
    "Faça o diagnóstico do problema de saúde do paciente com base nos sintomas e no protocolo interno "
    "fornecido (se houver), e sugira o tratamento recomendado. Nunca prescreva diretamente -- apenas "
    "sugira condutas para validação médica. Lembre sempre de destacar que é necessário validação médica."
)

def montar_input(dados_paciente, contexto_protocolo, pergunta_medico):
    partes = [f"Pergunta do medico: {pergunta_medico}"]
    if dados_paciente:
        partes.append(f"Dados do paciente: {json.dumps(dados_paciente, ensure_ascii=False)}")
    if contexto_protocolo:
        partes.append(f"Protocolo interno relevante: {contexto_protocolo}")
    return "\n".join(partes)

def gerar_sugestao(estado: EstadoAssistente) -> EstadoAssistente:
    if llm is not None:
        entrada = montar_input(
            estado.get("dados_paciente"),
            estado.get("contexto_protocolo", ""),
            estado.get("pergunta_medico", ""),
        )
        prompt = ALPACA_TEMPLATE.format(instruction=INSTRUCAO_DIAGNOSTICO, input=entrada)
        bruto = llm.invoke(prompt)
        texto = bruto.content if hasattr(bruto, "content") else str(bruto)
    else:
        snippet = (estado.get("contexto_protocolo", "") or "")[:200]
        texto = f"(offline) Baseado no protocolo disponivel: {snippet}..."

    log_auditoria({
        "etapa": "gerar_sugestao",
        "patient_id": estado["patient_id"],
        "usou_llm_finetuned": llm is not None,
    })
    return {"sugestao": texto}


Camada de segurança: um filtro por expressão regular procura padrões de dosagem/via de administração na resposta gerada. Se encontrar, marca `flag_revisao_obrigatoria=True` — sinal explícito de que um médico precisa validar antes de qualquer ação. O disclaimer de validação humana é sempre anexado, com ou sem a flag.

In [12]:
PADRAO_PRESCRICAO_DIRETA = re.compile(r"\b\d+\s?(mg|ml|mcg|g|comprimidos?|gotas?)\b", re.IGNORECASE)

DISCLAIMER = (
    "\n\nEsta é uma sugestão gerada por IA com base em protocolos internos."
    " NÃO substitui avaliação clínica. Validação de um médico responsável é obrigatória "
    "antes de qualquer conduta ou prescrição."
)

def validar_seguranca(estado: EstadoAssistente) -> EstadoAssistente:
    resposta = estado.get("sugestao", "")
    contem_dosagem = bool(PADRAO_PRESCRICAO_DIRETA.search(resposta))
    resposta_final = resposta.strip() + DISCLAIMER

    log_auditoria({
        "etapa": "validar_seguranca",
        "patient_id": estado["patient_id"],
        "flag_revisao_obrigatoria": contem_dosagem,
        "fontes": estado.get("fontes_protocolo"),
    })
    return {"sugestao": resposta_final, "flag_revisao_obrigatoria": contem_dosagem}


Rota alternativa: se a pergunta não for clínica (ex.: algo administrativo), o assistente não aciona o LLM nem o RAG — só devolve uma resposta padrão explicando os limites de atuação dele. Isso também é parte da camada de segurança: o assistente não tenta responder fora do escopo para o qual foi validado.

In [13]:
def resposta_fora_do_escopo(estado: EstadoAssistente) -> EstadoAssistente:
    log_auditoria({"etapa": "fora_de_escopo", "patient_id": estado["patient_id"]})
    return {
        "sugestao": (
            "Esta pergunta esta fora do escopo clínico deste assistente. "
            "Para questões administrativas, procure a secretaria ou o sistema de gestão hospitalar."
        ),
        "flag_revisao_obrigatoria": False,
    }


## 7) Montagem do grafo (arestas lineares e condicionais)

Fluxo: carregar paciente → verificar exames → classificar intenção → (condicional) dúvida clínica segue para buscar protocolo → gerar sugestão → validar segurança; fora de escopo vai direto para a resposta padrão.

In [14]:
g = StateGraph(EstadoAssistente)
g.add_node("carregar_paciente", carregar_paciente)
g.add_node("verificar_exames", verificar_exames_pendentes)
g.add_node("classificar_intencao", classificar_intencao)
g.add_node("buscar_protocolo", buscar_protocolo)
g.add_node("gerar_sugestao", gerar_sugestao)
g.add_node("validar_seguranca", validar_seguranca)
g.add_node("resposta_fora_do_escopo", resposta_fora_do_escopo)

g.set_entry_point("carregar_paciente")
g.add_edge("carregar_paciente", "verificar_exames")
g.add_edge("verificar_exames", "classificar_intencao")
g.add_conditional_edges("classificar_intencao", rota_intencao, {
    "duvida_clinica": "buscar_protocolo",
    "fora_de_escopo": "resposta_fora_do_escopo",
})
g.add_edge("buscar_protocolo", "gerar_sugestao")
g.add_edge("gerar_sugestao", "validar_seguranca")
g.add_edge("validar_seguranca", END)
g.add_edge("resposta_fora_do_escopo", END)

app = g.compile()
print(app.get_graph().draw_ascii())


                      +-----------+                         
                      | __start__ |                         
                      +-----------+                         
                            *                               
                            *                               
                            *                               
                  +-------------------+                     
                  | carregar_paciente |                     
                  +-------------------+                     
                            *                               
                            *                               
                            *                               
                  +------------------+                      
                  | verificar_exames |                      
                  +------------------+                      
                            *                               
                        

## 8) Execução de exemplos

Três casos: uma dúvida clínica de rotina, um paciente com exame pendente (deve gerar alerta) e uma pergunta fora do escopo clínico (deve cair na rota de recusa).

In [15]:
testes = [
    {"patient_id": "P001", "pergunta_medico": "Qual o protocolo de tratamento indicado para esse paciente com diabetes e possivel fratura?"},
    {"patient_id": "P001", "pergunta_medico": "Qual o diagnostico mais provavel para dor no femur apos queda?"},
    {"patient_id": "P002", "pergunta_medico": "Onde fica o setor de faturamento do hospital?"},
]

for t in testes:
    estado_inicial: EstadoAssistente = {
        "patient_id": t["patient_id"],
        "pergunta_medico": t["pergunta_medico"],
    }
    resultado = app.invoke(estado_inicial)

    print("\n=== Pergunta ===")
    print(t["pergunta_medico"])
    print("Paciente:", resultado.get("dados_paciente"))
    if resultado.get("alerta"):
        print("ALERTA PARA EQUIPE MEDICA:", resultado["alerta"])
    print("Intencao classificada:", resultado.get("intencao"))
    print("Sugestao:", resultado.get("sugestao"))
    print("Fontes usadas:", resultado.get("fontes_protocolo"))
    print("Revisao humana obrigatoria?:", resultado.get("flag_revisao_obrigatoria"))



=== Pergunta ===
Qual o protocolo de tratamento indicado para esse paciente com diabetes e possivel fratura?
Paciente: {'patient_id': 'P001', 'iniciais': 'J.S.', 'idade': 67, 'historico': 'Hipertensao, diabetes tipo 2', 'exames_pendentes': 'Hemograma completo', 'medicamentos_atuais': 'Metformina, Losartana'}
ALERTA PARA EQUIPE MEDICA: Exame pendente: Hemograma completo
Intencao classificada: duvida_clinica
Sugestao: Aumente os níveis glicêmicos até 140 mg/dL na hemoglobina glucosedada ou entre 100 e
180 mg / dL sem correções adicionais

Esta é uma sugestão gerada por IA com base em protocolos internos. NÃO substitui avaliação clínica. Validação de um médico responsável é obrigatória antes de qualquer conduta ou prescrição.
Fontes usadas: ['Diabete melito tipo 2.pdf']
Revisao humana obrigatoria?: True

=== Pergunta ===
Qual o diagnostico mais provavel para dor no femur apos queda?
Paciente: {'patient_id': 'P001', 'iniciais': 'J.S.', 'idade': 67, 'historico': 'Hipertensao, diabetes tipo